In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch import device
import logging

In [3]:
# Dataset class

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class UCSDDataset(Dataset):
    def __init__(self, root_dir, is_train=True, transform=None):
        self.root_dir = root_dir
        self.is_train = is_train
        self.transform = transform
        self.frames = []
        self.labels = []
        self.load_frames()

    def load_frames(self):
        if self.is_train:
            normal_dir = os.path.join(self.root_dir, 'Normal')
            anomaly_dir = os.path.join(self.root_dir, 'Anomaly')
            
            if not os.path.exists(normal_dir) or not os.path.exists(anomaly_dir):
                raise FileNotFoundError(f"Directory not found. Please check if the path is correct: {self.root_dir}")
            
            for folder in os.listdir(normal_dir):
                folder_path = os.path.join(normal_dir, folder)
                for frame in os.listdir(folder_path):
                    if frame.endswith('.tif'):
                        self.frames.append(os.path.join(folder_path, frame))
                        self.labels.append(0)  # Normal frame
            
            for folder in os.listdir(anomaly_dir):
                folder_path = os.path.join(anomaly_dir, folder)
                for frame in os.listdir(folder_path):
                    if frame.endswith('.tif'):
                        self.frames.append(os.path.join(folder_path, frame))
                        self.labels.append(1)  # Anomaly frame
        else:
            normal_dir = os.path.join(self.root_dir, 'Normal')
            anomaly_dir = os.path.join(self.root_dir, 'Anomaly')
            
            for folder in os.listdir(normal_dir):
                folder_path = os.path.join(normal_dir, folder)
                for frame in os.listdir(folder_path):
                    if frame.endswith('.tif'):
                        self.frames.append(os.path.join(folder_path, frame))
                        self.labels.append(0)  # Normal frame
            
            for folder in os.listdir(anomaly_dir):
                if not folder.endswith('_gt'):
                    folder_path = os.path.join(anomaly_dir, folder)
                    gt_folder_path = os.path.join(anomaly_dir, folder + '_gt')
                    for frame in os.listdir(folder_path):
                        if frame.endswith('.tif'):
                            self.frames.append(os.path.join(folder_path, frame))
                            gt_frame = os.path.join(gt_folder_path, frame)
                            if os.path.exists(gt_frame):
                                gt_img = Image.open(gt_frame).convert('L')
                                if np.any(np.array(gt_img) > 0):
                                    self.labels.append(1)  # Anomaly frame
                                else:
                                    self.labels.append(0)  # Normal frame
                            else:
                                self.labels.append(0)  # Assume normal if no gt

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, idx):
        frame_path = self.frames[idx]
        label = self.labels[idx]
        try:
            image = Image.open(frame_path).convert("RGB")
        except OSError as e:
            logger.error(f"Error loading image {frame_path}: {str(e)}")
            # Return a placeholder image or skip this sample
            return None, label
        if self.transform:
            image = self.transform(image)
        return image, label

def add_gaussian_noise(x):
    return x + 0.01 * torch.randn_like(x)

def collate_fn(batch):
    batch = list(filter(lambda x: x[0] is not None, batch))
    return torch.utils.data.dataloader.default_collate(batch)

# Transformations for the images
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomResizedCrop((128, 128), scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    add_gaussian_noise  # Adding Gaussian noise
])

train_dataset = UCSDDataset('UCSD_Anomaly_Dataset/UCSDped1/Train', is_train=True, transform=transform)
test_dataset = UCSDDataset('UCSD_Anomaly_Dataset/UCSDped1/Test', is_train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, collate_fn=collate_fn)

# Check if data loading works correctly
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")

Number of training samples: 8107
Number of testing samples: 6010


In [4]:
# Diffusion Model
class DiffusionModel(nn.Module):
    def __init__(self, time_steps=1000):
        super().__init__()
        self.time_steps = time_steps
        self.time_embed = nn.Embedding(time_steps, 256)
        
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            ResidualBlock(64, 128),
            nn.MaxPool2d(2),
            ResidualBlock(128, 256),
            nn.MaxPool2d(2),
            ResidualBlock(256, 512),
            nn.MaxPool2d(2),
            ResidualBlock(512, 512),
        )
        
        self.decoder = nn.Sequential(
            ResidualBlock(512 + 256, 512),
            nn.Upsample(scale_factor=2),
            ResidualBlock(512, 256),
            nn.Upsample(scale_factor=2),
            ResidualBlock(256, 128),
            nn.Upsample(scale_factor=2),
            ResidualBlock(128, 64),
            nn.Conv2d(64, 3, 3, padding=1)
        )
        
    def forward(self, x, t):
        t_embed = self.time_embed(t)
        t_embed = t_embed.view(t_embed.shape[0], t_embed.shape[1], 1, 1).expand(-1, -1, x.shape[2]//8, x.shape[3]//8)
        
        features = self.encoder(x)
        x_input = torch.cat([features, t_embed], dim=1)
        return self.decoder(x_input)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.relu = nn.ReLU()
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
    def forward(self, x):
        residual = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += self.shortcut(residual)
        return self.relu(x)

In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.cuda.current_device()}")
print(f"Device name: {torch.cuda.get_device_name(0)}")

Using device: cuda
CUDA available: True
Current device: 0
Device name: NVIDIA GeForce RTX 4050 Laptop GPU


In [7]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm

# Model, optimizer, loss function, and scheduler
model = DiffusionModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# Early stopping parameters
patience = 10
best_loss = float('inf')
patience_counter = 0

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    try:
        for batch, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            if len(batch) == 0:
                continue
            batch = batch.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            t = torch.randint(0, model.time_steps, (batch.shape[0],)).to(device)
            noise = torch.randn_like(batch)
            noisy_batch = batch + noise * torch.sqrt(t.view(-1, 1, 1, 1) / model.time_steps)
            predicted_noise = model(noisy_batch, t)
            loss = criterion(predicted_noise, noise)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
   
        avg_loss = train_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
        
        # Learning rate scheduler step
        scheduler.step(avg_loss)
        
        # Early stopping check
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            # Save the best model
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs")
                break
   
    except Exception as e:
        logger.error(f"Error in training loop: {str(e)}")
        continue

print("Training finished.")

c:\Users\mayan\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 1/100: 100%|██████████| 254/254 [07:23<00:00,  1.74s/it]


Epoch 1/100, Loss: 1.0017


Epoch 2/100: 100%|██████████| 254/254 [07:14<00:00,  1.71s/it]


Epoch 2/100, Loss: 0.9742


Epoch 3/100: 100%|██████████| 254/254 [07:32<00:00,  1.78s/it]


Epoch 3/100, Loss: 0.9369


... (132 outputs left out)

Epoch 70/100: 100%|██████████| 254/254 [07:14<00:00,  1.71s/it]


Epoch 70/100, Loss: 0.4751


Epoch 71/100:  57%|█████▋    | 144/254 [04:10<03:11,  1.74s/it]


KeyboardInterrupt: 

In [8]:
logger = logging.getLogger(__name__)

# Anomaly detection function
def detect_anomalies(model, test_batch):
    model.eval()
    with torch.no_grad():
        t = torch.ones(test_batch.shape[0]).long().to(device) * (model.time_steps - 1)
        for _ in range(model.time_steps):
            predicted_noise = model(test_batch, t)
            test_batch = (test_batch - predicted_noise * (1 / model.time_steps ** 0.5)).clamp(0, 1)
            t -= 1
            if t[0] < 0:
                break
    return criterion(test_batch, test_batch).mean(dim=[1,2,3])

# Evaluation
def evaluate_model(model, test_loader, device, criterion):
    model.eval()
    anomaly_scores = []
    ground_truth = []
    
    try:
        for batch, labels in tqdm(test_loader, desc="Evaluating"):
            if len(batch) == 0:
                continue
            batch = batch.to(device)
            labels = labels.to(device)
            scores = detect_anomalies(model, batch)
            anomaly_scores.extend(scores.cpu().numpy())
            ground_truth.extend(labels.cpu().numpy())
    
    except Exception as e:
        logger.error(f"Error in evaluation loop: {str(e)}")
        return None, None
    
    return np.array(anomaly_scores), np.array(ground_truth)

# Load the best model
best_model_path = 'best_model.pth'
model.load_state_dict(torch.load(best_model_path))
model.to(device)

# Run evaluation
anomaly_scores, ground_truth = evaluate_model(model, test_loader, device, criterion)

if anomaly_scores is not None and ground_truth is not None:
    # Calculate ROC and EER
    fpr, tpr, thresholds = roc_curve(ground_truth, anomaly_scores)
    roc_auc = roc_auc_score(ground_truth, anomaly_scores)
    
    # Calculate EER
    fnr = 1 - tpr
    eer_threshold = thresholds[np.nanargmin(np.absolute(fnr - fpr))]
    eer = fpr[np.nanargmin(np.absolute(fnr - fpr))]
    
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"EER: {eer:.4f}")
else:
    print("Evaluation failed due to an error.")

C:\Users\mayan\AppData\Local\Temp\ipykernel_20440\2716183835.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))
Evaluati

KeyboardInterrupt: 

In [ ]:
# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()